# Isolate whole-genome sequencing: plasmid copy number

Figure 4B

Environment: `python_env` (`envs/python_env.yml`)

Input: `data/WGS_plasmid_chromosome_coverage.csv`, mean chromosome and plasmid depth per isolate, from `preprocessing/WGS_coverage.py` (minimap2 alignments of the PRJNA1536727 reads to the ATCC 15597 reference)

In [ ]:
from pathlib import Path
import yaml
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

plt.rcParams["pdf.fonttype"] = 42

ROOT = Path("..")
config = yaml.safe_load(open(ROOT / "config.yaml"))
OUT_DIR = Path(config["out_dir"])
FIG_DIR = OUT_DIR / "figures"
SRC_DIR = OUT_DIR / "source_data"
FIG_DIR.mkdir(parents=True, exist_ok=True)
SRC_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df = pd.read_csv(ROOT / "data" / "WGS_plasmid_chromosome_coverage.csv")
df["ratio"] = df.plasmid_coverage / df.chrom_coverage
df["group"] = df["sample"].str.startswith("c").map({True: "Control", False: "Infected"})
df["num"] = df["sample"].str.extract(r"(\d+)", expand=False).astype(int)
df = df.sort_values(["group", "num"]).reset_index(drop=True)

test = ttest_ind(df.loc[df.group == "Control", "ratio"], df.loc[df.group == "Infected", "ratio"], equal_var=False)
test

## Figure 4B

In [ ]:
LABEL_FS, TICK_FS, ANNOT_FS = 22, 20, 18


def format_p(p):
    if p > 0.05:
        return "ns"
    return f"p = {p:.2e}" if p < 0.001 else f"p = {p:.3f}"


fig, ax = plt.subplots(figsize=(14, 6.5))
sns.barplot(data=df, x="sample", y="ratio", hue="group", order=df["sample"], dodge=False, ax=ax)
ax.axhline(1, color="black", linestyle="--", linewidth=1)
ax.set_xlabel("Sample", fontsize=LABEL_FS)
ax.set_ylabel("Plasmid / chromosome coverage", fontsize=LABEL_FS)
ax.tick_params(axis="x", rotation=90, labelsize=TICK_FS)
ax.tick_params(axis="y", labelsize=TICK_FS)
ax.text(0.98, 0.04, f"Welch's t-test\nControl vs infected: {format_p(test.pvalue)}",
        transform=ax.transAxes, fontsize=ANNOT_FS, fontfamily="monospace", ha="right", va="bottom",
        multialignment="left", bbox=dict(boxstyle="round,pad=0.5", facecolor="white", alpha=0.9, edgecolor="lightgray"))
ax.legend(title="Group", loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=21, title_fontsize=24,
          frameon=False, handlelength=1.6, handleheight=1.2, alignment="left")

plt.tight_layout()
fig.savefig(FIG_DIR / "Fig4B.png", dpi=300, bbox_inches="tight")
fig.savefig(FIG_DIR / "Fig4B.pdf", bbox_inches="tight")
plt.show()

df.drop(columns="num").to_csv(SRC_DIR / "Fig4B_data.csv", index=False)